In [1]:
from lib.helders import *
from lib.data_processing import *
from params import PARAMS

SYMBOL = 'SOLUSDC'

# Load data
blocks = load_from_pickle(PARAMS['base_dir'], SYMBOL)
print(f'Loaded {len(blocks)} blocks')

In [2]:
# Prototype logic (instant execution, no chase)
import numpy as np, random, matplotlib.pyplot as plt
from numba import njit

@njit
def run_proto(
        all_trades: np.ndarray,
        start_index: int,
        d_top_threshold: int,
        d_btm_threshold: int,
        d_top_target: int,
        d_btm_target: int,
        alloc: int = 1000,
    ) -> tuple[int, np.ndarray, tuple[float, float, float, float]]:
    """
    Simplified logic without timeouts, chase or volumes.
    Instant full execution on event.
       
       Input:
          - all_trades: numpy array of shape (N, 4)
          - start_index: index to start observation from
          - d_top_threshold: upper threshold offset (steps) from start price
          - d_btm_threshold: lower threshold offset (steps) from start price
          - d_top_target: upper target offset (steps) from start price
          - d_btm_target: lower target offset (steps) from start price

       Output:
          - stopped_at_index
          - groups: (g, 3) -> [start_index, end_index, reason, side]
            reasons: 1 open_long, 2 open_short, 3 close_long_target, 4 close_short_target,
                     5 close_long_neutral, 6 close_short_neutral
          - lines: (top_target, btm_target, top_threshold, btm_threshold)
    """
    base_price = all_trades[start_index, 1]

    top_threshold = base_price + d_top_threshold
    btm_threshold = base_price - d_btm_threshold
    top_target = base_price + d_top_target
    btm_target = base_price - d_btm_target

    # At most two groups (open, close)
    groups = np.zeros((alloc, 4), dtype=np.int64)
    groups_pos = 0
   
    # Run simplified logic
    position_side = 0
    stopped_at_index = start_index

    for i in range(start_index, all_trades.shape[0]):
        # Allocate more groups if needed
        if groups_pos + 1 == groups.shape[0]:
            groups = np.concatenate((groups, np.zeros((alloc, 4), dtype=np.int64)))

        # Current price
        price = all_trades[i, 1]

        if position_side == 0:
            if price > top_threshold:
                groups[groups_pos, 0] = i
                groups[groups_pos, 1] = top_threshold + 1  
                groups[groups_pos, 2] = 1 # open_long
                groups[groups_pos, 3] = 1 # buy order
                groups_pos += 1
                position_side = 1
            
            elif price < btm_threshold:
                groups[groups_pos, 0] = i
                groups[groups_pos, 1] = btm_threshold - 1
                groups[groups_pos, 2] = 2 # open_short
                groups[groups_pos, 3] = -1 # sell order
                groups_pos += 1
                position_side = -1
            
            continue

        if position_side == 1:
            if price >= top_target:
                groups[groups_pos, 0] = i
                groups[groups_pos, 1] = top_target
                groups[groups_pos, 2] = 3 # close_long_target
                groups[groups_pos, 3] = -1 # sell order
                groups_pos += 1
                stopped_at_index = i
                break
            
            elif price <= top_threshold:
                groups[groups_pos, 0] = i
                groups[groups_pos, 1] = top_threshold
                groups[groups_pos, 2] = 5 # close_long_neutral
                groups[groups_pos, 3] = -1 # sell order
                groups_pos += 1
                position_side = 0
            
            continue

        if position_side == -1:
            if price <= btm_target:
                groups[groups_pos, 0] = i
                groups[groups_pos, 1] = btm_target
                groups[groups_pos, 2] = 4 # close_short_target
                groups[groups_pos, 3] = 1 # buy order
                groups_pos += 1
                stopped_at_index = i
                break
            
            elif price >= btm_threshold:
                groups[groups_pos, 0] = i
                groups[groups_pos, 1] = btm_threshold
                groups[groups_pos, 2] = 6 # close_short_neutral
                groups[groups_pos, 3] = 1 # buy order
                groups_pos += 1
                position_side = 0
            
            continue
   
    # Pack lines
    lines = (
        top_target,
        btm_target,
        top_threshold,
        btm_threshold,
    )

    return stopped_at_index, groups[:groups_pos], lines

def plot_proto(
        all_trades: np.ndarray,
        start_index: int,
        stopped_at_index: int,
        groups: np.ndarray,
        d_top_threshold: int,
        d_btm_threshold: int,
        d_top_target: int,
        d_btm_target: int,
        s: int = 300,
    ) -> None:
    # Bounds
    f = max(0, start_index - s)
    t = min(all_trades.shape[0], max(stopped_at_index, start_index) + s)

    # Market data
    dates = [datetime.fromtimestamp(d/1000) for d in all_trades[f:t, 0]]
    prices = all_trades[f:t, 1]

    # Levels
    base_price = all_trades[start_index, 1]

    top_threshold = base_price + d_top_threshold
    btm_threshold = base_price - d_btm_threshold
    top_target = base_price + d_top_target
    btm_target = base_price - d_btm_target

    # Plot
    plt.clf()
    _, ax = plt.subplots(1, 1, figsize=(20, 8))

    ax.plot(dates, prices)
    ax.set_title('Proto logic')

    # Thresholds and targets
    ax.axhline(top_threshold, color='orange', linestyle='--', linewidth=1)
    ax.axhline(btm_threshold, color='orange', linestyle='--', linewidth=1)
    ax.axhline(top_target, color='green', linestyle=':', linewidth=1)
    ax.axhline(btm_target, color='red', linestyle=':', linewidth=1)

    # Mark groups
    for k in range(groups.shape[0]):
        gi = groups[k]
        gi_idx = int(gi[0])
        gi_price = all_trades[gi_idx, 1]
        gi_dt = datetime.fromtimestamp(all_trades[gi_idx, 0] / 1000)
        reason = int(gi[2])

        if reason == 1:
            ax.scatter([gi_dt], [gi_price], c='green', s=80, marker='^')
        elif reason == 2:
            ax.scatter([gi_dt], [gi_price], c='red', s=80, marker='v')
        elif reason == 3:
            ax.scatter([gi_dt], [gi_price], c='black', s=80, marker='o')
        elif reason == 4:
            ax.scatter([gi_dt], [gi_price], c='green', s=80, marker='s')
        elif reason == 5:
            ax.scatter([gi_dt], [gi_price], c='red', s=80, marker='s')

    # Window markers
    start_dt = datetime.fromtimestamp(all_trades[start_index, 0] / 1000)
    ax.axvline(start_dt, color='gray', linestyle='--', linewidth=1)

    if stopped_at_index >= 0 and stopped_at_index < all_trades.shape[0]:
        stop_dt = datetime.fromtimestamp(all_trades[stopped_at_index, 0] / 1000)
        ax.axvline(stop_dt, color='gray', linestyle='--', linewidth=1)

    # Plot lines
    ax.axhline(top_target, color='green', linestyle=':', linewidth=1)
    ax.axhline(btm_target, color='red', linestyle=':', linewidth=1)
    ax.axhline(top_threshold, color='orange', linestyle='--', linewidth=1)
    ax.axhline(btm_threshold, color='orange', linestyle='--', linewidth=1)

    plt.show()

def verbal_reason(reason: int) -> str:
    if reason == 1:
        return 'open_long'
    elif reason == 2:
        return 'open_short'
    elif reason == 3:
        return 'close_long_target'
    elif reason == 4:
        return 'close_short_target'
    elif reason == 5:
        return 'close_long_neutral'
    elif reason == 6:
        return 'close_short_neutral'
    else:
        raise ValueError(f'Unknown reason: {reason}')

# Proto example
# Random block
all_trades, price_step, volume_step = random.choice(blocks)

# Select start index
start_index = random.randint(0, all_trades.shape[0] - 1)

# Thresholds/targets in price steps (relative to price at start_index)
d_top_threshold = 1 # units
d_btm_threshold = 1
d_top_target_perc = 2 # percent
d_btm_target_perc = 2

# Base price for reference
base_price = all_trades[start_index, 1]
print(f'Base price (steps): {base_price}')

# Calculate targets in units
d_top_target = int(base_price * d_top_target_perc / 100)
d_btm_target = int(base_price * d_btm_target_perc / 100)
print(f'Top target (units): {d_top_target}')
print(f'Btm target (units): {d_btm_target}')

# Run
stopped_at_index, groups, lines = run_proto(
    all_trades,
    start_index,
    d_top_threshold,
    d_btm_threshold,
    d_top_target,
    d_btm_target,
)

print(f'Start index: {start_index}')
print(f'Stopped at index: {stopped_at_index} ({stopped_at_index - start_index} trades)')

# Show groups summary
print('Groups:')
base, quote = 0, 0
for i in range(groups.shape[0]):
    g = groups[i]
    b, p = g[3], g[1]
    base += b
    quote -= p * b
    print(f'  {i}: start={int(g[0])}, end={int(g[1])}, reason={int(g[2])} ({verbal_reason(int(g[2]))}), base={base}, quote={quote}')

# Last quote = PnL
pnl = 100 * quote / all_trades[0, 1]
print(f'PnL: {pnl:.4f}%')

# Relative PnL
relative_pnl = pnl / len(groups)
print(f'Relative PnL: {relative_pnl:.4f}%')

# Plot
plot_proto(
    all_trades,
    start_index,
    stopped_at_index,
    groups,
    d_top_threshold,
    d_btm_threshold,
    d_top_target,
    d_btm_target,
    s=500,
)


In [3]:
# Quick mass test
d_top_threshold = 1 # units
d_btm_threshold = 1
d_top_target_perc = 2 # percent
d_btm_target_perc = 2

n_tests = 10_000

ok, count, diff = 0, 0, 0
d_map = {}
pbar = tqdm(total=n_tests, desc='Testing')

for _ in range(n_tests):
    # Random block
    all_trades, price_step, volume_step = random.choice(blocks)

    # Select start index
    start_index = random.randint(0, all_trades.shape[0] - 1)

    # Base price for reference
    base_price = all_trades[start_index, 1]

    # Calculate targets in units
    d_top_target = int(base_price * d_top_target_perc / 100)
    d_btm_target = int(base_price * d_btm_target_perc / 100)

    # Run
    stopped_at_index, groups, lines = run_proto(
        all_trades,
        start_index,
        d_top_threshold,
        d_btm_threshold,
        d_top_target,
        d_btm_target,
    )

    if stopped_at_index == start_index:
        continue

    base, quote = 0, 0
    for i in range(groups.shape[0]):
        g = groups[i]
        b, p = g[3], g[1]
        base += b
        quote -= p * b
   
    # Last quote = PnL
    pnl = 100 * quote / all_trades[0, 1]
   
    # Relative PnL
    d = pnl / len(groups)

    # Add to stats
    ok += 1 if d > 0 else 0
    count += 1
    diff += d

    t = int(all_trades[start_index, 0] / 60_000) * 60_000
    if t not in d_map:
        d_map[t] = 0

    d_map[t] += d

    pbar.update(1)

pbar.close()

# Stats
print(f'SR: {ok / count:.4f}, Avg. diff: {diff / count:.4f}%, Sum. diff: {diff:.4f}%')

# Plot
date, line = [], []
s = 0

for t in sorted(d_map.keys()):
    s += d_map[t]
    date.append(datetime.fromtimestamp(t / 1000))
    line.append(s)

plt.clf()
plt.figure(figsize=(20, 8))

plt.plot(date, line)
plt.title('Cumulative PnL')
plt.xlabel('Time (min)')
plt.ylabel('PnL (%)')
plt.show()


In [4]:
# Parametrized test (prototype)
from tqdm import tqdm
from lib.indicators import *

# Plot pairs
def np_sma(a, period):
    sma = np.full(len(a), np.nan)

    for i in range(period - 1, len(a)):
        sma[i] = np.mean(a[i - period + 1:i + 1])

    return sma

def plot_pairs(pairs, title):
    pairs.sort(key=lambda x: x[0])

    x = np.array([p[0] for p in pairs])
    y = np.array([p[1] for p in pairs])

    y = np_sma(y, max(2, int(len(y) / 50)))

    plt.clf()
    plt.figure(figsize=(16, 8))
    plt.plot(x, y)
    plt.title(title)
    plt.show()

# Params
d_top_threshold = 1 # units
d_btm_threshold = 1
tf_ms = 300_000
period = 30

n_tests = 100_000

for top, btm in [(5, 5), (5, 0.5), (0.5, 5)]:
    d_top_target_perc = 5 # percent
    d_btm_target_perc = 0.5
    
    print(f'Checking {top} / {btm}')
    
    results = []
    pbar = tqdm(total=n_tests, desc='Testing')

    for _ in range(n_tests):
        pbar.update(1)
        
        # Random block
        all_trades, price_step, volume_step = random.choice(blocks)

        # Select start index
        start_index = random.randint(0, all_trades.shape[0] - 1)

        # Base price for reference
        base_price = all_trades[start_index, 1]

        # Calculate targets in units
        d_top_target = int(base_price * d_top_target_perc / 100)
        d_btm_target = int(base_price * d_btm_target_perc / 100)

        # Run
        stopped_at_index, groups, lines = run_proto(
            all_trades,
            start_index,
            d_top_threshold,
            d_btm_threshold,
            d_top_target,
            d_btm_target,
        )

        if stopped_at_index == start_index:
            continue

        base, quote = 0, 0
        for i in range(groups.shape[0]):
            g = groups[i]
            b, p = g[3], g[1]
            base += b
            quote -= p * b
    
        # Last quote = PnL
        pnl = 100 * quote / all_trades[0, 1]
    
        # Relative PnL
        d = pnl / len(groups)

        # Parametrize
        t, v, e = get_trend_and_volatility_indicator(all_trades, start_index, tf_ms, period)
        if e != 0:
            continue

        results.append((t, v, d))

    pbar.close()

    title = f'{top} / {btm}'

    plot_pairs([(t, d) for t, v, d in results], title=f'{title} (trend, PnL)')
    plot_pairs([(v, d) for t, v, d in results], title=f'{title} (volatility, PnL)')